# Notebook 01 - Events (Week 1)

Load Person 1's JSONL events, understand every column, count by `event_type`,
list missing values, group by `incident_id`, and plot activity over time.

**Deliverable:** you can explain every column (see `docs/glossary.md`).

Run the generator first: `python data_synthetic/generate_events.py`

In [ ]:
import sys
from pathlib import Path


def _package_root() -> Path:
    for base in [Path.cwd(), *Path.cwd().parents]:
        if (base / 'sentinelllm' / 'analysis').is_dir():
            return base / 'sentinelllm'
        if base.name == 'sentinelllm' and (base / 'analysis').is_dir():
            return base
    raise RuntimeError('Run this notebook from inside the sentinelllm/ tree')


PACKAGE_ROOT = _package_root()
if str(PACKAGE_ROOT) not in sys.path:
    sys.path.insert(0, str(PACKAGE_ROOT))

import matplotlib.pyplot as plt
import pandas as pd

from analysis.load_events import (
    event_type_counts,
    incident_summary,
    load_events,
    load_labels,
    missing_values,
)

pd.set_option('display.max_columns', 40)
print('package root:', PACKAGE_ROOT)

## Load the events and the label manifest

In [ ]:
events = load_events()
labels = load_labels()

print('events:', events.shape)
print('incidents:', events['incident_id'].nunique())
print('columns:')
for column in events.columns:
    print('  -', column)

## Count events by `event_type`

In [ ]:
event_type_counts(events)

## Missing values

Most NaNs are expected: a `network_connect` event has no `process.*` fields, a
`file_create` has no `network.*` fields, and so on. They are not data errors.
If a *common* field is unexpectedly missing, raise it with Person 1.

In [ ]:
missing = missing_values(events)
missing[missing > 0].to_frame('missing')

## Group events by incident

In [ ]:
summary = incident_summary(events)
print('label distribution (incidents):')
print(summary['label'].value_counts().to_string())
summary.head(10)

## One incident in detail

Pick any incident and read its events in timestamp order - this is the raw
material the LLM analyst and the dashboard will later reconstruct.

In [ ]:
sample_id = 'INC-0065' if 'INC-0065' in set(events['incident_id']) else events['incident_id'].iloc[0]
incident = events[events['incident_id'] == sample_id].sort_values('timestamp')
incident[['event_id', 'timestamp', 'event_type', 'rule_hits']]

## Activity over time

In [ ]:
fig, ax = plt.subplots(figsize=(10, 3))
events.set_index('timestamp').resample('6h').size().plot(ax=ax, color='steelblue')
ax.set_title('Events over time (6h bins)')
ax.set_xlabel('time (UTC)')
ax.set_ylabel('events')
plt.tight_layout()
plt.show()

## Week 1 checklist

- [x] Load JSONL into a DataFrame
- [x] Count events by `event_type`
- [x] List missing values
- [x] Group events by `incident_id`
- [x] Plot counts over time
- [ ] Walk through every column with `docs/glossary.md` and confirm you can explain it